# Redes Neuronales Convolucionales Avanzadas: Desafíos Actuales y Arquitecturas Modernas

## Contenido
1. **LeNet-5** en MNIST (repaso rápido)
2. **CNN personalizada** en CIFAR-10
3. **Funciones de pérdida modernas** y sus desafíos
4. **Transfer Learning** con ImageNet (VGG16, ResNet50)
5. **Arquitecturas modernas**: MobileNet, YOLOv8, EfficientNet
6. **Guardado y carga de pesos**
7. **Desafíos actuales** en deep learning

## 1. LeNet-5 en MNIST (Repaso)

LeNet-5 es una de las primeras CNN, diseñada para reconocimiento de dígitos manuscritos.

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models, datasets
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Cargar MNIST
(x_train, y_train), (x_test, y_test) = datasets.mnist.load_data()
x_train = x_train.reshape(-1, 28, 28, 1) / 255.0
x_test = x_test.reshape(-1, 28, 28, 1) / 255.0

# LeNet-5 adaptado a MNIST
def create_lenet():
    model = models.Sequential([
        layers.Conv2D(6, (5,5), activation='tanh', input_shape=(28,28,1), padding='same'),
        layers.AveragePooling2D((2,2), strides=2),
        layers.Conv2D(16, (5,5), activation='tanh'),
        layers.AveragePooling2D((2,2), strides=2),
        layers.Conv2D(120, (5,5), activation='tanh'),
        layers.Flatten(),
        layers.Dense(84, activation='tanh'),
        layers.Dense(10, activation='softmax')
    ])
    return model

lenet = create_lenet()
lenet.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
lenet.summary()

# Entrenamiento rápido (solo 3 épocas para demostración)
history_lenet = lenet.fit(x_train, y_train, epochs=3, validation_split=0.1, verbose=1)
lenet.evaluate(x_test, y_test, verbose=0)
print(f"LeNet-5 Accuracy MNIST: {lenet.evaluate(x_test, y_test, verbose=0)[1]:.4f}")

h:\Anaconda\envs\deepf\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 28, 28, 6)      │           156 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d               │ (None, 14, 14, 6)      │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 10, 10, 16)     │         2,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_1             │ (None, 5, 5, 16)       │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 1, 1, 120)      │        48,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 120)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 84)             │        10,164 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           850 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 61,706 (241.04 KB)

 Trainable params: 61,706 (241.04 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/3
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9299 - loss: 0.2351 - val_accuracy: 0.9747 - val_loss: 0.0943
Epoch 2/3
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.9739 - loss: 0.0849 - val_accuracy: 0.9810 - val_loss: 0.0654
Epoch 3/3
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.9815 - loss: 0.0583 - val_accuracy: 0.9823 - val_loss: 0.0588
LeNet-5 Accuracy MNIST: 0.9802


## 2. CNN Personalizada en CIFAR-10

CIFAR-10 es más desafiante que MNIST: imágenes pequeñas (32x32) pero a color (3 canales).

In [2]:
# Cargar CIFAR-10
(x_train_cifar, y_train_cifar), (x_test_cifar, y_test_cifar) = datasets.cifar10.load_data()
x_train_cifar = x_train_cifar / 255.0
x_test_cifar = x_test_cifar / 255.0

# Arquitectura CNN mejorada para CIFAR-10
def create_cifar_cnn():
    model = models.Sequential([
        layers.Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(32,32,3)),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2,2)),
        layers.Dropout(0.2),
        
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2,2)),
        layers.Dropout(0.3),
        
        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2,2)),
        layers.Dropout(0.4),
        
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(10, activation='softmax')
    ])
    return model

cifar_cnn = create_cifar_cnn()
cifar_cnn.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
cifar_cnn.summary()

# Entrenamiento (solo 5 épocas para demostración)
history_cifar = cifar_cnn.fit(x_train_cifar, y_train_cifar, epochs=5, validation_split=0.1, verbose=1)
test_loss, test_acc = cifar_cnn.evaluate(x_test_cifar, y_test_cifar, verbose=0)
print(f"CNN Custom CIFAR-10 Accuracy: {test_acc:.4f}")

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 285s 2us/step


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 32, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 32, 32, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 32, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 16, 16, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 8, 8, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 8, 8, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 8, 8, 128)      │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 8, 8, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 4, 4, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 4, 4, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 816,938 (3.12 MB)

 Trainable params: 815,530 (3.11 MB)

 Non-trainable params: 1,408 (5.50 KB)

Epoch 1/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 84s 56ms/step - accuracy: 0.4428 - loss: 1.6333 - val_accuracy: 0.5954 - val_loss: 1.1244
Epoch 2/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 75s 54ms/step - accuracy: 0.6218 - loss: 1.0701 - val_accuracy: 0.6656 - val_loss: 0.9569
Epoch 3/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 79s 56ms/step - accuracy: 0.6823 - loss: 0.9089 - val_accuracy: 0.7354 - val_loss: 0.7454
Epoch 4/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 74s 52ms/step - accuracy: 0.7212 - loss: 0.8049 - val_accuracy: 0.7700 - val_loss: 0.6619
Epoch 5/5
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 74s 53ms/step - accuracy: 0.7487 - loss: 0.7354 - val_accuracy: 0.7660 - val_loss: 0.6687
CNN Custom CIFAR-10 Accuracy: 0.7626


## 3. Funciones de Pérdida Modernas y Desafíos

### Funciones de pérdida comunes:

| Función | Fórmula | Uso típico |
|---------|---------|------------|
| Cross-Entropy | $L = -\sum y_i \log(\hat{y}_i)$ | Clasificación |
| Focal Loss | $L = -\alpha(1-p_t)^\gamma \log(p_t)$ | Clases desbalanceadas |
| Dice Loss | $L = 1 - \frac{2|A \cap B|}{|A|+|B|}$ | Segmentación |
| Triplet Loss | $L = \max(d(a,p) - d(a,n) + \alpha, 0)$ | Aprendizaje de embeddings |
| Contrastive Loss | $L = y d^2 + (1-y) \max(m - d, 0)^2$ | Siamesas |

### Desafíos actuales:
1. **Overfitting**: El modelo memoriza en lugar de generalizar
2. **Underfitting**: El modelo no aprende lo suficiente
3. **Vanishing/Exploding gradients**: Gradientes muy pequeños o grandes
4. **Class imbalance**: Clases con muy pocos ejemplos
5. **Domain shift**: Los datos de prueba difieren de entrenamiento
6. **Catastrophic forgetting**: Al aprender nuevas tareas, olvida las anteriores

In [3]:
# Ejemplo: Focal Loss para clases desbalanceadas
import tensorflow as tf

def focal_loss(gamma=2.0, alpha=0.25):
    def focal_loss_fixed(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, tf.keras.backend.epsilon(), 1 - tf.keras.backend.epsilon())
        cross_entropy = -y_true * tf.math.log(y_pred)
        weight = alpha * y_true * tf.math.pow(1 - y_pred, gamma)
        loss = weight * cross_entropy
        return tf.reduce_mean(loss)
    return focal_loss_fixed

print("Focal Loss definida para manejar clases desbalanceadas")

Focal Loss definida para manejar clases desbalanceadas


## 4. Transfer Learning con ImageNet

El **Transfer Learning** permite usar pesos pre-entrenados en ImageNet (1.2M imágenes, 1000 clases) para tareas con pocos datos.

In [4]:
from tensorflow.keras.applications import VGG16, ResNet50, EfficientNetB0, MobileNetV2
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg_preprocess

# Cargar modelo pre-entrenado (sin la capa superior)
# Nota: Para CIFAR-10, las imágenes son muy pequeñas, pero demostramos el concepto

# VGG16
base_vgg = VGG16(weights='imagenet', include_top=False, input_shape=(32,32,3))
base_vgg.trainable = False  # Congelar pesos pre-entrenados

# ResNet50
# base_resnet = ResNet50(weights='imagenet', include_top=False, input_shape=(32,32,3))
# base_resnet.trainable = False

# MobileNetV2 (más liviano)
base_mobilenet = MobileNetV2(weights='imagenet', include_top=False, input_shape=(32,32,3))
base_mobilenet.trainable = False

# Crear modelo completo
def create_transfer_model(base_model, num_classes=10):
    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

# Modelo con MobileNetV2 (más rápido para demostración)
transfer_model = create_transfer_model(base_mobilenet)
transfer_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
transfer_model.summary()

print("Modelo de transfer learning creado - los pesos de MobileNetV2 están congelados")
print("En la práctica, se fine-tune después de entrenar la cabeza")

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 35s 1us/step


C:\Users\yoda\AppData\Local\Temp\ipykernel_33632\1214215606.py:16: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_mobilenet = MobileNetV2(weights='imagenet', include_top=False, input_shape=(32,32,3))


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 1, 1, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,588,490 (9.87 MB)

 Trainable params: 330,506 (1.26 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

Modelo de transfer learning creado - los pesos de MobileNetV2 están congelados
En la práctica, se fine-tune después de entrenar la cabeza


## 5. Arquitecturas Modernas

### VGG16
- Arquitectura simple: solo capas convolucionales 3x3 y max pooling
- 138 millones de parámetros, profunda pero simple
- Buena para transfer learning

### ResNet (Residual Networks)
- Introduce conexiones residuales (skip connections)
- Soluciona el problema de degradación en redes muy profundas
- Fórmula: $y = F(x) + x$

### MobileNet
- Usa **depthwise separable convolutions** para ser liviana
- Ideal para dispositivos móviles y edge computing

### EfficientNet
- Escala uniformemente profundidad, ancho y resolución
- Mejor rendimiento con menos parámetros

### YOLO (You Only Look Once)
- Detección de objetos en tiempo real
- YOLOv8 es la versión más reciente
- Divide la imagen en grillas y predice bounding boxes

In [5]:
# Comparación de arquitecturas (código conceptual)

architectures = {
    'VGG16': '138M params, 16 capas, muy profunda pero simple',
    'ResNet50': '25M params, conexiones residuales, 50 capas',
    'MobileNetV2': '3.5M params, separable convolutions, para móviles',
    'EfficientNetB0': '5.3M params, scaling uniforme, estado del arte',
    'YOLOv8': 'Detección en tiempo real, anchor-free, 75+ FPS'
}

for name, desc in architectures.items():
    print(f"{name}: {desc}")

# Ejemplo de carga de YOLOv8 (requiere ultralytics)
# from ultralytics import YOLO
# model = YOLO('yolov8n.pt')  # nano version
# results = model('image.jpg')

VGG16: 138M params, 16 capas, muy profunda pero simple
ResNet50: 25M params, conexiones residuales, 50 capas
MobileNetV2: 3.5M params, separable convolutions, para móviles
EfficientNetB0: 5.3M params, scaling uniforme, estado del arte
YOLOv8: Detección en tiempo real, anchor-free, 75+ FPS


## 6. Guardado y Carga de Pesos

Es fundamental guardar los modelos entrenados para reutilizarlos o continuar entrenamiento.

In [6]:
# Guardar modelo completo (arquitectura + pesos)
cifar_cnn.save('cifar_cnn_model.h5')
print("Modelo guardado como 'cifar_cnn_model.h5'")

# Guardar solo los pesos
cifar_cnn.save_weights('cifar_cnn_weights.h5')
print("Pesos guardados como 'cifar_cnn_weights.h5'")

# Cargar modelo completo
# loaded_model = tf.keras.models.load_model('cifar_cnn_model.h5')

# Cargar pesos en una arquitectura existente
# new_model = create_cifar_cnn()
# new_model.load_weights('cifar_cnn_weights.h5')

# Formato más nuevo (SavedModel)
cifar_cnn.export('saved_model')
print("Modelo exportado en formato SavedModel")

Modelo guardado como 'cifar_cnn_model.h5'


ValueError: The filename must end in `.weights.h5`. Received: filepath=cifar_cnn_weights.h5

## 7. Desafíos Actuales en Deep Learning

### 1. Interpretabilidad (XAI - Explainable AI)
- **Problema**: Las redes son cajas negras
- **Soluciones**: LIME, SHAP, Grad-CAM, Attention Maps

### 2. Datos limitados
- **Problema**: Requieren grandes cantidades de datos
- **Soluciones**: Data augmentation, Transfer Learning, Few-shot learning

### 3. Robustez y adversarial attacks
- **Problema**: Pequeñas perturbaciones engañan a la red
- **Soluciones**: Adversarial training, defensas certificadas

### 4. Sesgo y equidad (Fairness)
- **Problema**: Modelos heredan sesgos de los datos
- **Soluciones**: Balanced datasets, fairness constraints

### 5. Eficiencia computacional
- **Problema**: Modelos muy grandes (billones de parámetros)
- **Soluciones**: Pruning, Quantization, Knowledge Distillation

### 6. Aprendizaje continuo (Continual Learning)
- **Problema**: Catastrophic forgetting
- **Soluciones**: EWC, Memory replay, Progressive networks

### 7. Generalización fuera de distribución (OOD)
- **Problema**: Fallan cuando los datos de prueba difieren
- **Soluciones**: Domain adaptation, Domain generalization

In [ ]:
# Ejemplo: Data Augmentation para mejorar generalización
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])

# Modelo con data augmentation integrada
def create_model_with_augmentation():
    inputs = tf.keras.Input(shape=(32,32,3))
    x = data_augmentation(inputs)
    x = layers.Conv2D(32, (3,3), activation='relu')(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Conv2D(64, (3,3), activation='relu')(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation='relu')(x)
    outputs = layers.Dense(10, activation='softmax')(x)
    return tf.keras.Model(inputs, outputs)

print("Data augmentation mejora la generalización y reduce overfitting")

## Resumen

| Arquitectura | Año | Parámetros | Característica clave |
|--------------|-----|------------|---------------------|
| LeNet-5 | 1998 | 60k | Base de CNNs |
| AlexNet | 2012 | 60M | ReLU, Dropout, GPU |
| VGG16 | 2014 | 138M | Muy profunda, simple |
| ResNet50 | 2015 | 25M | Conexiones residuales |
| MobileNetV2 | 2018 | 3.5M | Liviana para móviles |
| EfficientNet | 2019 | 5.3M | Scaling uniforme |
| YOLOv8 | 2023 | - | Detección en tiempo real |

### Recomendaciones prácticas:
1. **Para clasificación básica**: Usar MobileNet o EfficientNet pre-entrenadas
2. **Para detección**: YOLOv8 o Faster R-CNN
3. **Para segmentación**: U-Net, Mask R-CNN
4. **Con pocos datos**: Transfer Learning + Fine-tuning
5. **Para producción**: Quantization + Pruning para reducir tamaño